# Stage 1 · Supervised U-Net on KITTI

Select a GPU runtime in Colab and run the cells in order. Uses the paired improved-depth Eigen subset from Kaggle. If Kaggle requests authentication, configure it in this runtime and rerun the download cell.

Scenes or drives are held out as groups. This is a project split for comparing U-Net and ViT, so save the exact split manifest. The test split is evaluated after checkpoint selection. Both datasets use supervised masked MAE in metres.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import json
import math
import re
import shutil
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 42
EPOCHS, PATIENCE, BATCH_SIZE = 30, 6, 4
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
HEIGHT, WIDTH = 192, 640
MAX_DEPTH = 80.0
NAME = "kitti"
DATA_ROOT = Path("/content/stage1/datasets/kitti")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
def find_pairs(root):
    drive_pattern = re.compile(r"\d{4}_\d{2}_\d{2}_drive_\d+_sync")

    def index_frames(paths, prefer_png=False):
        frames = {}
        for path in sorted(paths):
            drive = next((p.name for p in path.parents if drive_pattern.fullmatch(p.name)), None)
            if drive is None:
                continue
            key = (drive, path.stem)
            previous = frames.get(key)
            if previous is None:
                frames[key] = path
            elif prefer_png and previous.parent == path.parent and previous.suffix.lower() != path.suffix.lower():
                if path.suffix.lower() == ".png":
                    frames[key] = path
            else:
                raise ValueError(f"Duplicate frame {key}: {previous} and {path}")
        return frames

    rgb = index_frames(
        (p for p in root.glob("**/image_02/data/*")
         if p.suffix.lower() in {".png", ".jpg", ".jpeg"}),
        prefer_png=True,
    )
    depth = index_frames(root.glob("**/proj_depth/groundtruth/image_02/*.png"))
    return [(rgb[key], depth[key], key[0]) for key in sorted(rgb.keys() & depth.keys())]



pairs = find_pairs(DATA_ROOT)
if not pairs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    subprocess.run(["kaggle", "datasets", "download", "-d",
                    "aalniak/kitti-eigen-split-for-monocular-depth-estimation",
                    "--unzip", "-p", str(DATA_ROOT)], check=True)
    pairs = find_pairs(DATA_ROOT)
if not pairs:
    raise RuntimeError("No paired KITTI RGB and depth files were found.")
records = [(f"{drive}/{rgb.stem}", drive) for rgb, _, drive in pairs]

def read_pair(index):
    rgb_path, depth_path, _ = pairs[index]
    with Image.open(rgb_path) as image:
        rgb = image.convert("RGB")
    with Image.open(depth_path) as image:
        depth = Image.fromarray(np.asarray(image, dtype=np.float32) / 256.0)
    if rgb.size != depth.size:
        raise ValueError(f"Unaligned KITTI pair: {rgb_path} and {depth_path}")
    return rgb, depth

In [ ]:
groups = np.array(sorted({group for _, group in records}))
assert len(groups) >= 7, "Need enough independent scenes or drives for three splits."
np.random.default_rng(SEED).shuffle(groups)
held_out = max(1, round(len(groups) * 0.15))
group_parts = {
    "test": set(groups[:held_out]),
    "validation": set(groups[held_out:2 * held_out]),
    "train": set(groups[2 * held_out:]),
}
indices = {
    name: [i for i, (_, group) in enumerate(records) if group in members]
    for name, members in group_parts.items()
}
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output = Path("/content/stage1/results") / NAME / run_id
output.mkdir(parents=True)
manifest = {
    name: [{"index": i, "sample": records[i][0], "group": records[i][1]} for i in rows]
    for name, rows in indices.items()
}
(output / "splits.json").write_text(json.dumps(manifest, indent=2))
print({name: len(rows) for name, rows in indices.items()})

class PairedDepth(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, item):
        rgb, depth = read_pair(self.rows[item])
        rgb = rgb.resize((WIDTH, HEIGHT), Image.Resampling.BILINEAR)
        depth = depth.resize((WIDTH, HEIGHT), Image.Resampling.NEAREST)
        x = torch.from_numpy(np.array(rgb, dtype=np.float32)).permute(2, 0, 1) / 255
        y = torch.from_numpy(np.array(depth, dtype=np.float32)).unsqueeze(0)
        return x, y

datasets = {name: PairedDepth(rows) for name, rows in indices.items()}
loaders = {
    name: DataLoader(ds, batch_size=BATCH_SIZE, shuffle=(name == "train"),
                     generator=torch.Generator().manual_seed(SEED))
    for name, ds in datasets.items()
}

In [ ]:
class UNetDepth(nn.Module):
    def __init__(self):
        super().__init__()

        def block(inputs, outputs):
            return nn.Sequential(
                nn.Conv2d(inputs, outputs, 3, padding=1), nn.ReLU(),
                nn.Conv2d(outputs, outputs, 3, padding=1), nn.ReLU(),
            )

        self.pool = nn.MaxPool2d(2)
        self.enc1 = block(3, 16)
        self.enc2 = block(16, 32)
        self.enc3 = block(32, 64)
        self.bridge = block(64, 128)
        self.dec3 = block(128 + 64, 64)
        self.dec2 = block(64 + 32, 32)
        self.dec1 = block(32 + 16, 16)
        self.head = nn.Conv2d(16, 1, 1)

    def forward(self, rgb):
        a = self.enc1(rgb)
        b = self.enc2(self.pool(a))
        c = self.enc3(self.pool(b))
        value = self.bridge(self.pool(c))
        for skip, decode in ((c, self.dec3), (b, self.dec2), (a, self.dec1)):
            value = F.interpolate(value, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            value = decode(torch.cat((value, skip), dim=1))
        return F.softplus(self.head(value))

In [ ]:
def evaluate_epoch(network, batches, optimizer=None):
    network.train(optimizer is not None)
    sums = torch.zeros(4, dtype=torch.float64, device=device)
    count = 0
    with torch.set_grad_enabled(optimizer is not None):
        for rgb, depth in batches:
            rgb, depth = rgb.to(device), depth.to(device)
            valid = torch.isfinite(depth) & (depth > 0) & (depth <= MAX_DEPTH)
            observed = depth[valid]
            if observed.numel() == 0:
                continue
            predicted = network(rgb)[valid]
            error = (predicted - observed).abs()
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True)
                error.mean().backward()
                optimizer.step()
            with torch.no_grad():
                sums += torch.stack((
                    error.sum(dtype=torch.float64),
                    error.square().sum(dtype=torch.float64),
                    (error / observed).sum(dtype=torch.float64),
                    (torch.maximum(predicted / observed, observed / predicted) < 1.25).sum(),
                ))
                count += observed.numel()
    if count == 0:
        raise ValueError("No valid depth pixels.")
    mae, mse, abs_rel, delta1 = (sums / count).cpu().tolist()
    if not all(math.isfinite(value) for value in (mae, mse, abs_rel, delta1)):
        raise FloatingPointError("Non-finite evaluation metrics.")
    return dict(mae_m=mae, rmse_m=math.sqrt(mse), abs_rel=abs_rel, delta1=delta1, pixels=count)

model = UNetDepth().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
best_mae, stale, best_epoch = float("inf"), 0, 0
checkpoint = output / "unet_best.pt"
with (output / "history.csv").open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=["epoch", "train_mae_m", "val_mae_m", "val_rmse_m"])
    writer.writeheader()
    for epoch in range(1, EPOCHS + 1):
        train = evaluate_epoch(model, loaders["train"], optimizer)
        val = evaluate_epoch(model, loaders["validation"])
        writer.writerow(dict(epoch=epoch, train_mae_m=train["mae_m"],
                             val_mae_m=val["mae_m"], val_rmse_m=val["rmse_m"]))
        stream.flush()
        print(f"Epoch {epoch:02d} | train MAE {train['mae_m']:.3f} m | validation MAE {val['mae_m']:.3f} m")
        if val["mae_m"] < best_mae:
            best_mae, stale, best_epoch = val["mae_m"], 0, epoch
            torch.save(model.state_dict(), checkpoint)
        else:
            stale += 1
            if stale >= PATIENCE:
                break

model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
model.eval()

In [ ]:
report = {
    "model": "U-Net", "dataset": NAME, "seed": SEED, "best_epoch": best_epoch,
    "height": HEIGHT, "width": WIDTH, "max_depth_m": MAX_DEPTH,
    "epochs": EPOCHS, "patience": PATIENCE, "batch_size": BATCH_SIZE,
    "training_loss": "masked MAE in meters", "device": str(device),
    "parameters": sum(p.numel() for p in model.parameters()),
    "validation": evaluate_epoch(model, loaders["validation"]),
    "test": evaluate_epoch(model, loaders["test"]),
}
sample = datasets["test"][0][0].unsqueeze(0).to(device)
times = []
with torch.inference_mode():
    for i in range(30):
        if device.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        model(sample)
        if device.type == "cuda":
            torch.cuda.synchronize()
        if i >= 10:
            times.append((time.perf_counter() - start) * 1000)
report["median_forward_ms"] = float(np.median(times))
(output / "metrics.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

In [ ]:
for name, dataset in datasets.items():
    for i in np.linspace(0, len(dataset) - 1, min(3, len(dataset)), dtype=int):
        rgb, depth = dataset[i]
        with torch.inference_mode():
            estimate = model(rgb.unsqueeze(0).to(device))[0, 0].cpu().numpy()
        measured = depth[0].numpy()
        measured = np.ma.masked_where(
            ~np.isfinite(measured) | (measured <= 0) | (measured > MAX_DEPTH), measured
        )
        fig, axes = plt.subplots(1, 3, figsize=(12, 4), layout="constrained")
        axes[0].imshow(rgb.permute(1, 2, 0))
        axes[1].imshow(measured, cmap="plasma", vmin=0, vmax=MAX_DEPTH)
        heatmap = axes[2].imshow(estimate, cmap="plasma", vmin=0, vmax=MAX_DEPTH)
        for axis, title in zip(axes, ["RGB", "Reference depth", "U-Net depth"]):
            axis.set_title(title)
            axis.axis("off")
        fig.colorbar(heatmap, ax=axes[1:], label="Depth (m)", extend="max")
        fig.savefig(output / f"{name}_{i:04d}.png", dpi=140)
        plt.show()
        plt.close(fig)

from google.colab import files
archive = shutil.make_archive(str(output), "zip", root_dir=output)
files.download(archive)